# 🔭 Laboratorio: Benchmark Auto-ARIMA Walk-Forward\n
**Objetivo:** Ejecutar simulaciones `Walk-Forward` masivas utilizando Auto-ARIMA como modelo de referencia tradicional.\n
**Métricas:** MAPE, Hit_Local, CumHit_B (Direccionalidad).\n
**Archivo de Salida:** `macro_backtest_arima_db.csv` (Anexión Segura / Checkpointing).

In [2]:
import sys
import os
import time
import warnings
sys.path.append(os.path.abspath('..'))

# Ocultar warnings matemáticos durante la simulación masiva\n
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

from src.ui.market_loader import MarketLoader
from src.quant_engine.arima_evaluator import AutoARIMAWalkForwardEvaluator

### 1. Configuración de la Batería de Pruebas
Aquí se define el universo de activos y las temporalidades a auditar.

In [3]:
# 1. Definir los Tickers a analizar
tickers = ['MSFT', 'XLF', 'C', 'BTC-USD', 'SPY', 'AAPL', 'ETH-USD', 'JPM', 'BAC', 'MA', 'AMZN' ]

# 2. Definir los escenarios de tiempo
# Nota: Se configuró '2y' para '1d' para evitar bloqueos por límite de horas en YFinance.
timeframes = [
    {'period': '5y', 'interval': '1d'}
]

db_path = "backtest_arima_.csv"

# 3. Manejador de DB y Checkpointing (Idempotencia)
processed_configs = set()
if os.path.exists(db_path):
    df_existente = pd.read_csv(db_path)
    if not df_existente.empty:
        # Agrupar por Ticker, Periodo, Intervalo para saber qué combinaciones ya terminaron
        agrupado = df_existente.groupby(['Ticker', 'Periodo_Historia', 'Intervalo_Velas']).size().reset_index()
        for _, row in agrupado.iterrows():
            processed_configs.add((row['Ticker'], row['Periodo_Historia'], row['Intervalo_Velas']))

print(f"⚙️ Total Combinaciones Posibles en el Universo: {len(tickers) * len(timeframes)}")
print(f"📂 Combinaciones ya procesadas y guardadas en DB: {len(processed_configs)}\n")

⚙️ Total Combinaciones Posibles en el Universo: 11
📂 Combinaciones ya procesadas y guardadas en DB: 0



### 2. Motor de Ejecución Masiva
Este proceso puede tardar horas. Si lo detienes, la próxima vez que lo inicies continuará donde se quedó.

In [4]:
for ticker in tickers:
    for tf in timeframes:
        period = tf['period']
        interval = tf['interval']
        config_key = (ticker, period, interval)
        
        if config_key in processed_configs:
            print(f"⏭️ Omitiendo {ticker} [{period} | {interval}] - Ya existe en la base de datos.")
            continue
            
        print(f"\n🚀 Iniciando simulación para {ticker} [{period} | {interval}]...")
        
        # --- DESCARGA CON PROTECCIÓN ---
        try:
            df_mercado = MarketLoader.load_ticker_data(ticker, period=period, interval=interval)
        except Exception as e:
            print(f"❌ Error descargando datos para {ticker} ({period}/{interval}): {e}")
            continue
            
        total_velas = len(df_mercado)
        if total_velas < 200:
            print(f"⚠️ Omitiendo {ticker} [{period} | {interval}]: Historial demasiado corto ({total_velas} velas).")
            continue
            
        # --- LÓGICA DE SALTOS DINÁMICOS ---
        # Evita que 1 hora sature el procesador con miles de iteraciones redundantes
        SALTO = 150 if interval == '1h' else 20
        VENTANA_INICIAL = 150
        HORIZONTE = 30
        BLOQUES = 15
        
        print(f"   ► Velas Disponibles: {total_velas} | Salto Iterativo: {SALTO} | Predicción a Ciegas: {HORIZONTE}")
        
        # Instanciamos el Motor (Modo Financiero Riguroso: Log-Retornos)
        evaluador = AutoARIMAWalkForwardEvaluator(df_mercado, disable_norm=False, disable_returns=False)
        
        try:
            # Ejecutar el Auto-Tuner Iterativo en el tiempo
            df_resultados = evaluador.run(initial_window=VENTANA_INICIAL, stride=SALTO, horizon=HORIZONTE, blocks=BLOQUES)
            
            # Extraer resultados brutos (Incluyendo fallas extremas para ser estadísticamente honestos)
            df_guardar = df_resultados.copy()
            
            # INYECCIÓN DE METADATOS PARA EL FUTURO QUERY NOTEBOOK
            df_guardar.insert(0, 'Total_Velas_Disponible', total_velas)
            df_guardar.insert(0, 'Intervalo_Velas', interval)
            df_guardar.insert(0, 'Periodo_Historia', period)
            df_guardar.insert(0, 'Ticker', ticker)
            
            # Checkpoint Progressivo al CSV
            file_exists = os.path.isfile(db_path)
            df_guardar.to_csv(db_path, mode='a', header=not file_exists, index=False)
            
            print(f"✅ Éxito. Guardadas {len(df_guardar)} iteraciones para {ticker} en {db_path}.")
            
            # Registrar para que no se repita en caso de caída posterior en este mismo loop
            processed_configs.add(config_key)
            
            # Breve pausa para limpiar I/O
            time.sleep(1)
            
        except Exception as e:
            print(f"❌ Error matemático/sistémico durante el Backtest de {ticker}: {e}")
            continue


🚀 Iniciando simulación para MSFT [5y | 1d]...
   ► Velas Disponibles: 1255 | Salto Iterativo: 20 | Predicción a Ciegas: 30
Iniciando Auto-ARIMA Walk-Forward (Ventana:150, Salto:20, Horizonte:30 velas en 15 bloques)
✅ Éxito. Guardadas 54 iteraciones para MSFT en backtest_arima_.csv.

🚀 Iniciando simulación para XLF [5y | 1d]...
   ► Velas Disponibles: 1255 | Salto Iterativo: 20 | Predicción a Ciegas: 30
Iniciando Auto-ARIMA Walk-Forward (Ventana:150, Salto:20, Horizonte:30 velas en 15 bloques)
✅ Éxito. Guardadas 54 iteraciones para XLF en backtest_arima_.csv.

🚀 Iniciando simulación para C [5y | 1d]...
   ► Velas Disponibles: 1255 | Salto Iterativo: 20 | Predicción a Ciegas: 30
Iniciando Auto-ARIMA Walk-Forward (Ventana:150, Salto:20, Horizonte:30 velas en 15 bloques)
✅ Éxito. Guardadas 54 iteraciones para C en backtest_arima_.csv.

🚀 Iniciando simulación para BTC-USD [5y | 1d]...
   ► Velas Disponibles: 1827 | Salto Iterativo: 20 | Predicción a Ciegas: 30
Iniciando Auto-ARIMA Walk-For

### 3. Sanidad del Dataset
Lectura rápida para asegurar que la DB se está llenando correctamente.

In [5]:
if os.path.exists(db_path):
    df_final = pd.read_csv(db_path)
    print(f"📊 Total Filas en la Base de Datos: {len(df_final)}")
    display(df_final.head())
    display(df_final.tail())
else:
    print("El archivo CSV aún no ha sido creado.")

📊 Total Filas en la Base de Datos: 652


,Ticker,Periodo_Historia,Intervalo_Velas,Total_Velas_Disponible,Iteracion (Velas Vistas),Drift (k),SINDy R2,Validez,MAPE_B1,Naive_MAPE_B1,...,Hit_B13,CumHit_B13,MAPE_B14,Naive_MAPE_B14,Hit_B14,CumHit_B14,MAPE_B15,Naive_MAPE_B15,Hit_B15,CumHit_B15
0,MSFT,5y,1d,1255,150,0.0,1.0,OK,0.65,0.65,...,0.0,0.0,2.55,2.55,0.0,0.0,0.73,0.73,0.0,0.0
1,MSFT,5y,1d,1255,170,0.0,1.0,OK,2.99,2.99,...,0.0,0.0,7.11,7.11,0.0,0.0,12.66,12.66,0.0,0.0
2,MSFT,5y,1d,1255,190,0.0,1.0,OK,2.38,2.38,...,0.0,0.0,2.86,2.86,0.0,0.0,3.49,3.49,0.0,0.0
3,MSFT,5y,1d,1255,210,0.0,1.0,OK,0.55,0.55,...,0.0,0.0,0.62,0.62,0.0,0.0,2.49,2.49,0.0,0.0
4,MSFT,5y,1d,1255,230,0.0,1.0,OK,3.83,3.83,...,0.0,0.0,8.42,8.42,0.0,0.0,10.57,10.57,0.0,0.0


,Ticker,Periodo_Historia,Intervalo_Velas,Total_Velas_Disponible,Iteracion (Velas Vistas),Drift (k),SINDy R2,Validez,MAPE_B1,Naive_MAPE_B1,...,Hit_B13,CumHit_B13,MAPE_B14,Naive_MAPE_B14,Hit_B14,CumHit_B14,MAPE_B15,Naive_MAPE_B15,Hit_B15,CumHit_B15
647,AMZN,5y,1d,1255,1130,0.0,1.0,OK,1.28,1.28,...,0.0,0.0,9.77,9.77,0.0,0.0,12.10,12.10,0.0,0.0
648,AMZN,5y,1d,1255,1150,0.0,1.0,OK,2.06,2.06,...,0.0,0.0,2.33,2.33,0.0,0.0,3.76,3.76,0.0,0.0
649,AMZN,5y,1d,1255,1170,0.0,1.0,OK,1.07,1.07,...,0.0,0.0,21.19,21.19,0.0,0.0,22.97,22.97,0.0,0.0
650,AMZN,5y,1d,1255,1190,0.0,1.0,OK,2.08,2.08,...,0.0,0.0,6.00,6.00,0.0,0.0,1.31,1.31,0.0,0.0
651,AMZN,5y,1d,1255,1210,0.0,1.0,OK,2.77,2.77,...,0.0,0.0,8.40,8.40,0.0,0.0,7.08,7.08,0.0,0.0
